# LLM Counselor Evaluation - Memory NOT Included

This notebook evaluates LLM-generated therapeutic responses WITHOUT access to mem0 memories.

## Key Differences from Human Counselor Evaluation

- **Input**: Patient queries from transcript (NOT human counselor responses)
- **Response Source**: Generated by LLM using CBT_SYSTEM_PROMPT
- **What's Evaluated**: LLM-generated counselor responses
- **Memory Access**: NO - counselor only sees sliding window context
- **Baseline**: Fixed professional template (NOT first response)

## Features
- **Checkpointing**: Resume from any turn or transcript if interrupted
- **Markdown Logging**: All evaluations, scores, and reasoning saved to markdown
- **Output Organization**: All outputs saved to `./output_llm_counselor_memnotincluded/`

## Workflow

For each patient turn:
1. Add patient turn to mem0
2. Get conversation context (sliding window)
3. Generate LLM counselor response (NO memories)
4. Add LLM response to mem0
5. Evaluate CBT adherence
6. Evaluate persona consistency

In [ ]:
# Cell 1: Imports and Output Directory Setup
import sys
import os
import json
import time
import re
from pathlib import Path
from datetime import datetime
from dataclasses import asdict

sys.path.append(os.path.join(os.getcwd(), "our-pipeline"))

from openai import OpenAI
from mem0 import Memory
from transcript_parser import (
    parse_html_transcript_file,
    parse_html_transcript_text,
    get_conversation_context,
    get_counselor_turns,
    get_patient_turns,
    ConversationTurn
)
from mem0_integration import (
    create_mem0_config_with_llm,
    initialize_mem0,
    add_conversation_turn_to_memory,
    get_all_memories,
    audit_memories
)
from alignment_evaluators import (
    evaluate_cbt_adherence,
    evaluate_persona_consistency
)
from llm_counselor import generate_counselor_response
from therapeutic_framework import PROFESSIONAL_BASELINE_RESPONSE

# Output directory setup
OUTPUT_DIR = Path("./output_llm_counselor_memnotincluded")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "images").mkdir(exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)
(OUTPUT_DIR / "results").mkdir(exist_ok=True)

print("All modules loaded successfully!")
print(f"Output directory: {OUTPUT_DIR}")
print(f"  - Images: {OUTPUT_DIR}/images/")
print(f"  - Checkpoints: {OUTPUT_DIR}/checkpoints/")
print(f"  - Results: {OUTPUT_DIR}/results/")

In [28]:
# Cell 2: Configuration

# ============================================================================
# MODEL CONFIGURATION - Choose your backend
# ============================================================================

# OPTION A: Use Ollama (local, free)
USE_OLLAMA = False
OLLAMA_MODEL = "gpt-oss:20b"

# OPTION B: Use Lambda Cloud GPU instance
USE_LAMBDA_CLOUD = True
LAMBDA_CLOUD_BASE_URL = "http://localhost:11435/v1"  # Port 11435 for second Lambda instance
# OR use direct connection:
# LAMBDA_CLOUD_BASE_URL = "http://209.20.158.92:11434/v1"
LAMBDA_CLOUD_MODEL = "gpt-oss:20b"

# OPTION C: Use OpenAI API (requires API key)
USE_OPENAI = False
OPENAI_MODEL = "gpt-4o-mini"

# Separate model configuration for counselor vs judge
# (Will be set based on backend choice below)

print(f"Configuration:")
if USE_LAMBDA_CLOUD:
    COUNSELOR_MODEL = LAMBDA_CLOUD_MODEL
    JUDGE_MODEL = LAMBDA_CLOUD_MODEL
    print(f"  Backend: Lambda Cloud GPU")
    print(f"  Base URL: {LAMBDA_CLOUD_BASE_URL}")
    print(f"  Counselor Model: {COUNSELOR_MODEL}")
    print(f"  Judge Model: {JUDGE_MODEL}")
    print("  Make sure SSH tunnel is active: ssh -L 11435:localhost:11434 ubuntu@209.20.158.92")
elif USE_OLLAMA:
    COUNSELOR_MODEL = OLLAMA_MODEL
    JUDGE_MODEL = OLLAMA_MODEL
    print(f"  Backend: Ollama (local)")
    print(f"  Counselor Model: {COUNSELOR_MODEL}")
    print(f"  Judge Model: {JUDGE_MODEL}")
elif USE_OPENAI:
    COUNSELOR_MODEL = OPENAI_MODEL
    JUDGE_MODEL = OPENAI_MODEL
    print(f"  Backend: OpenAI API")
    print(f"  Counselor Model: {COUNSELOR_MODEL}")
    print(f"  Judge Model: {JUDGE_MODEL}")
else:
    raise ValueError("Please set one of USE_OLLAMA, USE_LAMBDA_CLOUD, or USE_OPENAI to True")

print(f"  Memory Access: NO")

Configuration:
  Backend: Lambda Cloud GPU
  Base URL: http://localhost:11435/v1
  Counselor Model: gpt-oss:20b
  Judge Model: gpt-oss:20b
  Make sure SSH tunnel is active: ssh -L 11435:localhost:11434 ubuntu@209.20.158.92
  Memory Access: NO


In [29]:
# Cell 3: Initialize OpenAI Client

if USE_LAMBDA_CLOUD:
    client = OpenAI(
        base_url=LAMBDA_CLOUD_BASE_URL,
        api_key="lambda"  # Ollama on Lambda doesn't need a real key
    )
    MODEL = LAMBDA_CLOUD_MODEL
    print(f"Using Lambda Cloud GPU instance")
    print(f"  Base URL: {LAMBDA_CLOUD_BASE_URL}")
    print(f"  Model: {MODEL}")
elif USE_OLLAMA:
    client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"
    )
    MODEL = OLLAMA_MODEL
    print(f"Using Ollama with model: {MODEL}")
elif USE_OPENAI:
    client = OpenAI()  # Uses OPENAI_API_KEY from environment
    MODEL = OPENAI_MODEL
    print(f"Using OpenAI with model: {MODEL}")

print("\nClient created successfully!")

Using Lambda Cloud GPU instance
  Base URL: http://localhost:11435/v1
  Model: gpt-oss:20b

Client created successfully!


In [ ]:
# Cell 4: Load the COMBINED transcript file for patient 0518-014
from pathlib import Path
import re

COMBINED_TRANSCRIPT_PATH = Path("./0518-014_combined_transcript.txt")

print(f"Loading combined transcript: {COMBINED_TRANSCRIPT_PATH}")
print("=" * 60)

# Read the combined transcript and split by transcript sections
with open(COMBINED_TRANSCRIPT_PATH, 'r', encoding='utf-8') as f:
    combined_content = f.read()

# Split by transcript headers (e.g., "========== 1000056544.txt ==========")
transcript_pattern = r'==========\s*(\d+\.txt)\s*=========='
sections = re.split(transcript_pattern, combined_content)

# Parse sections: alternates between content and filename
transcript_sections = []
current_filename = None
for i, section in enumerate(sections):
    if re.match(r'\d+\.txt', section.strip()):
        current_filename = section.strip()
    elif current_filename and section.strip():
        transcript_sections.append({
            'filename': current_filename,
            'content': section
        })
        current_filename = None

print(f"Found {len(transcript_sections)} transcript sections in combined file:")
for idx, ts in enumerate(transcript_sections, 1):
    print(f"  {idx}. {ts['filename']}")

# Parse ALL turns from the combined transcript, tracking source file
all_turns = []
turn_to_transcript_map = {}
transcript_boundaries = []

global_turn_number = 0
for ts in transcript_sections:
    try:
        section_turns = parse_html_transcript_text(ts['content'])
    except ValueError as e:
        print(f"  Warning: Could not parse {ts['filename']}: {e}")
        continue
    
    start_turn = global_turn_number + 1
    
    for turn in section_turns:
        global_turn_number += 1
        turn.turn_number = global_turn_number
        turn_to_transcript_map[global_turn_number] = ts['filename']
        all_turns.append(turn)
    
    end_turn = global_turn_number
    if end_turn >= start_turn:
        transcript_boundaries.append({
            'filename': ts['filename'],
            'start_turn': start_turn,
            'end_turn': end_turn,
            'turn_count': end_turn - start_turn + 1
        })

print(f"\nTotal turns across all transcripts: {len(all_turns)}")
print(f"Counselor turns: {len(get_counselor_turns(all_turns))}")
print(f"Patient turns: {len(get_patient_turns(all_turns))}")

print("\nTranscript boundaries:")
for tb in transcript_boundaries:
    print(f"  {tb['filename']}: turns {tb['start_turn']}-{tb['end_turn']} ({tb['turn_count']} turns)")

In [ ]:
# Cell 5: Initialize Mem0
import shutil
import time

RESET_MEMORIES = False  # Set to True for fresh run, False to keep existing memories (use False to resume)

# ChromaDB configuration - UNIQUE path for this notebook
CHROMA_DB_PATH = "./chroma_db_llm_counselor_memnotincluded"
CHROMA_COLLECTION_NAME = "llm_counselor_memnotincluded"

# Delete existing ChromaDB folder if reset is requested
# Handle Windows file locking issues gracefully
if RESET_MEMORIES and Path(CHROMA_DB_PATH).exists():
    try:
        shutil.rmtree(CHROMA_DB_PATH)
        print(f"Deleted existing {CHROMA_DB_PATH} folder for fresh start")
    except PermissionError as e:
        print(f"Warning: Could not delete {CHROMA_DB_PATH} - files may be locked by another process.")
        print(f"Error: {e}")
        print("To fix: Close any other notebooks/processes using this database, or restart your Python kernel.")
        print("Continuing with existing database...")
        RESET_MEMORIES = False  # Fall back to not resetting

# Unified USER_ID for persistent memory across all sessions of same patient
USER_ID = "patient_0518_014"

if USE_LAMBDA_CLOUD:
    # Get base URL without /v1 for Mem0
    mem0_base_url = LAMBDA_CLOUD_BASE_URL.replace("/v1", "")
    print(f"Mem0 base URL: {mem0_base_url}")
    
    mem_config = create_mem0_config_with_llm(
        llm_provider="ollama",  # Lambda runs Ollama
        model=LAMBDA_CLOUD_MODEL,
        base_url=mem0_base_url
    )
    mem_config["vector_store"]["config"]["collection_name"] = CHROMA_COLLECTION_NAME
    mem_config["vector_store"]["config"]["path"] = CHROMA_DB_PATH
    
    # Debug: print embedder config to verify URL
    print(f"Embedder config: {mem_config['embedder']}")
    
    memory = initialize_mem0(
        config=mem_config,
        reset_collection=RESET_MEMORIES
    )
    print(f"Initialized Mem0 with Lambda Cloud LLM: {LAMBDA_CLOUD_MODEL}")
elif USE_OLLAMA:
    mem_config = create_mem0_config_with_llm(
        llm_provider="ollama",
        model=OLLAMA_MODEL,
        base_url="http://localhost:11434"
    )
    mem_config["vector_store"]["config"]["collection_name"] = CHROMA_COLLECTION_NAME
    mem_config["vector_store"]["config"]["path"] = CHROMA_DB_PATH
    
    memory = initialize_mem0(
        config=mem_config,
        reset_collection=RESET_MEMORIES
    )
    print(f"Initialized Mem0 with Ollama LLM: {OLLAMA_MODEL}")
elif USE_OPENAI:
    memory = Memory()
    print(f"Initialized Mem0 with OpenAI")
    
if RESET_MEMORIES:
    print(f"  Reset memories for user: {USER_ID}")
print(f"Collection: {CHROMA_COLLECTION_NAME}")
print(f"ChromaDB Path: {CHROMA_DB_PATH}")
print(f"USER_ID: {USER_ID} (unified across all sessions)")

In [ ]:
# Cell 6: Main Processing Loop with Checkpoints and Markdown Logging

# ============================================================================
# CONFIGURATION
# ============================================================================
DELAY_BETWEEN_CALLS = 0.1 if (USE_OLLAMA or USE_LAMBDA_CLOUD) else 0.5  # Rate limiting (seconds)
RESUME_FROM_CHECKPOINT = True
VERBOSE = True  # Set to True for detailed turn-by-turn logging
baseline_response = PROFESSIONAL_BASELINE_RESPONSE

# ============================================================================
# CHECKPOINT AND MARKDOWN LOGGING FUNCTIONS
# ============================================================================

def get_checkpoint_path():
    """Get checkpoint file path for the combined transcript."""
    return OUTPUT_DIR / "checkpoints" / "combined_transcript_checkpoint.json"

def get_markdown_path():
    """Get markdown log path for the combined transcript."""
    return OUTPUT_DIR / "combined_transcript_evaluation_log.md"

def load_checkpoint():
    """Load checkpoint if exists."""
    checkpoint_path = get_checkpoint_path()
    if checkpoint_path.exists() and RESUME_FROM_CHECKPOINT:
        with open(checkpoint_path, 'r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        print(f"  Loaded checkpoint: {checkpoint['last_patient_turn_idx']} patient turns completed")
        return checkpoint
    return None

def save_checkpoint(checkpoint_data):
    """Save checkpoint to disk."""
    checkpoint_path = get_checkpoint_path()
    with open(checkpoint_path, 'w', encoding='utf-8') as f:
        json.dump(checkpoint_data, f, indent=2, ensure_ascii=False)

def init_markdown_log(total_turns, patient_count, boundaries):
    """Initialize markdown log file with transcript section info."""
    md_path = get_markdown_path()
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(f"# LLM Counselor Evaluation Log: Combined Transcript (Patient 0518-014)\n\n")
        f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write(f"**Counselor Model:** {COUNSELOR_MODEL}\n\n")
        f.write(f"**Judge Model:** {JUDGE_MODEL}\n\n")
        f.write(f"**Memory Enhanced:** No (memories NOT passed to LLM counselor)\n\n")
        f.write(f"**Context Source:** Simulated conversation (patient + LLM responses only)\n\n")
        f.write(f"**USER_ID:** {USER_ID} (unified across all sessions)\n\n")
        f.write(f"## Combined Transcript Info\n\n")
        f.write(f"- Total Turns: {total_turns}\n")
        f.write(f"- Patient Turns to Process: {patient_count}\n")
        f.write(f"- Number of Sessions: {len(boundaries)}\n\n")
        f.write(f"### Session Boundaries\n\n")
        f.write(f"| Session | Transcript | Turn Range | Turn Count |\n")
        f.write(f"|---------|------------|------------|------------|\n")
        for i, tb in enumerate(boundaries, 1):
            f.write(f"| {i} | {tb['filename']} | {tb['start_turn']}-{tb['end_turn']} | {tb['turn_count']} |\n")
        f.write(f"\n---\n\n")
        f.write(f"## Turn-by-Turn Evaluations\n\n")

def get_session_for_turn(turn_number, boundaries):
    """Get the session number and filename for a given turn."""
    for i, tb in enumerate(boundaries, 1):
        if tb['start_turn'] <= turn_number <= tb['end_turn']:
            return i, tb['filename']
    return None, None

def append_session_header_to_markdown(session_num, filename, start_turn, end_turn):
    """Append a session header to markdown when entering a new session."""
    md_path = get_markdown_path()
    with open(md_path, 'a', encoding='utf-8') as f:
        f.write(f"\n---\n\n")
        f.write(f"## Session {session_num}: {filename}\n\n")
        f.write(f"**Turns {start_turn} - {end_turn}**\n\n")
        f.write(f"---\n\n")

def append_turn_to_markdown(turn_number, patient_query, llm_response,
                            cbt_score, cbt_reasoning, persona_score, persona_reasoning,
                            memory_count, new_memories_this_turn, source_transcript):
    """Append a single turn evaluation to markdown log with full details."""
    md_path = get_markdown_path()
    with open(md_path, 'a', encoding='utf-8') as f:
        f.write(f"### Turn {turn_number} ({source_transcript})\n\n")
        f.write(f"**Patient Query:**\n> {patient_query[:500]}{'...' if len(patient_query) > 500 else ''}\n\n")
        f.write(f"**LLM Counselor Response:**\n> {llm_response[:500]}{'...' if len(llm_response) > 500 else ''}\n\n")
        f.write(f"**CBT Adherence Score:** {cbt_score}/10\n\n")
        f.write(f"**CBT Reasoning:**\n> {cbt_reasoning}\n\n")
        f.write(f"**Persona Consistency Score:** {persona_score}/10\n\n")
        f.write(f"**Persona Reasoning:**\n> {persona_reasoning}\n\n")
        f.write(f"**Memory Stats:** (not used in generation)\n")
        f.write(f"- Total memories accumulated: {memory_count}\n")
        f.write(f"- New memories this turn: {len(new_memories_this_turn)}\n\n")
        if new_memories_this_turn:
            f.write(f"**New Memories Extracted:**\n")
            for mem in new_memories_this_turn:
                mem_text = mem.get("memory", mem.get("text", str(mem)))
                metadata = mem.get("metadata", {})
                role = metadata.get("role", "?")
                turn = metadata.get("turn_number", "?")
                f.write(f"- `[Turn {turn}, {role}]` {mem_text[:200]}{'...' if len(str(mem_text)) > 200 else ''}\n")
            f.write("\n")
        f.write(f"---\n\n")

def append_memories_to_markdown(memories):
    """Append complete memory dump to markdown log."""
    md_path = get_markdown_path()
    with open(md_path, 'a', encoding='utf-8') as f:
        f.write(f"## Complete Memory Dump\n\n")
        f.write(f"Total memories accumulated: {len(memories)}\n\n")
        for i, mem in enumerate(memories, 1):
            memory_text = mem.get("memory", mem.get("text", str(mem)))
            metadata = mem.get("metadata", {})
            turn = metadata.get('turn_number', '?')
            role = metadata.get('role', '?')
            f.write(f"{i}. **[Turn {turn}, {role}]** {memory_text}\n\n")

def append_summary_to_markdown(cbt_results, persona_results, memory_count, boundaries):
    """Append summary statistics to markdown log with per-session breakdown."""
    md_path = get_markdown_path()
    cbt_scores = [r["score"] for r in cbt_results]
    persona_scores = [r["score"] for r in persona_results]
    with open(md_path, 'a', encoding='utf-8') as f:
        f.write(f"## Summary Statistics\n\n")
        f.write(f"### Overall CBT Adherence\n\n")
        f.write(f"- Mean: {sum(cbt_scores)/len(cbt_scores):.2f}/10\n")
        f.write(f"- Min: {min(cbt_scores)}/10\n")
        f.write(f"- Max: {max(cbt_scores)}/10\n\n")
        f.write(f"### Overall Persona Consistency\n\n")
        f.write(f"- Mean: {sum(persona_scores)/len(persona_scores):.2f}/10\n")
        f.write(f"- Min: {min(persona_scores)}/10\n")
        f.write(f"- Max: {max(persona_scores)}/10\n\n")
        f.write(f"### Memory\n\n")
        f.write(f"- Total Memories Stored: {memory_count}\n\n")
        
        # Per-session breakdown
        f.write(f"### Per-Session Statistics\n\n")
        f.write(f"| Session | Transcript | CBT Mean | Persona Mean | Evaluations |\n")
        f.write(f"|---------|------------|----------|--------------|-------------|\n")
        for i, tb in enumerate(boundaries, 1):
            session_cbt = [r["score"] for r in cbt_results if tb['start_turn'] <= r["turn_number"] <= tb['end_turn']]
            session_persona = [r["score"] for r in persona_results if tb['start_turn'] <= r["turn_number"] <= tb['end_turn']]
            if session_cbt:
                cbt_mean = sum(session_cbt) / len(session_cbt)
                persona_mean = sum(session_persona) / len(session_persona)
                f.write(f"| {i} | {tb['filename']} | {cbt_mean:.2f} | {persona_mean:.2f} | {len(session_cbt)} |\n")

def truncate(text, length=80):
    """Truncate text for display."""
    return text[:length] + "..." if len(text) > length else text

def serialize_simulated_turns(simulated_turns):
    """Convert simulated_turns to JSON-serializable format."""
    return [
        {
            "turn_number": t.turn_number,
            "role": t.role,
            "content": t.content,
            "timestamp": t.timestamp
        }
        for t in simulated_turns
    ]

def deserialize_simulated_turns(turns_data):
    """Convert JSON data back to ConversationTurn objects."""
    return [
        ConversationTurn(
            turn_number=t["turn_number"],
            role=t["role"],
            content=t["content"],
            timestamp=t.get("timestamp", "")
        )
        for t in turns_data
    ]

# ============================================================================
# MAIN PROCESSING LOOP - COMBINED TRANSCRIPT
# ============================================================================

patient_turns = get_patient_turns(all_turns)

print(f"Starting LLM Counselor evaluation (Memory NOT Included)")
print(f"Counselor Model: {COUNSELOR_MODEL}")
print(f"Judge Model: {JUDGE_MODEL}")
print(f"Mode: Memory NOT Included")
print(f"Total turns: {len(all_turns)} ({len(patient_turns)} patient turns)")
print(f"Sessions: {len(transcript_boundaries)}")
print(f"Resume from checkpoint: {RESUME_FROM_CHECKPOINT}")
print(f"Unified USER_ID: {USER_ID}")
print(f"Context Source: Simulated conversation (patient + LLM responses only)")
print("=" * 60)

# Load checkpoint if exists
checkpoint = load_checkpoint()

if checkpoint:
    cbt_results = checkpoint.get('cbt_results', [])
    persona_results = checkpoint.get('persona_results', [])
    generated_responses = checkpoint.get('generated_responses', [])
    memory_snapshots = checkpoint.get('memory_snapshots', [])
    last_patient_turn_idx = checkpoint.get('last_patient_turn_idx', 0)
    last_session_logged = checkpoint.get('last_session_logged', 0)
    # Restore simulated_turns from checkpoint
    simulated_turns_data = checkpoint.get('simulated_turns', [])
    simulated_turns = deserialize_simulated_turns(simulated_turns_data)
else:
    cbt_results = []
    persona_results = []
    generated_responses = []
    memory_snapshots = []
    last_patient_turn_idx = 0
    last_session_logged = 0
    simulated_turns = []  # Track patient turns + LLM responses (no human counselor)
    init_markdown_log(len(all_turns), len(patient_turns), transcript_boundaries)

# Track previous memories for detecting new memories per turn
previous_memory_ids = set()
initial_memories = get_all_memories(memory, USER_ID)
for mem in initial_memories:
    previous_memory_ids.add(mem.get("id", str(mem)))
print(f"Starting with {len(initial_memories)} existing memories")

remaining_turns = patient_turns[last_patient_turn_idx:]
print(f"Patient turns to process: {len(remaining_turns)} (starting from idx {last_patient_turn_idx})")

current_session = last_session_logged

for i, patient_turn in enumerate(remaining_turns):
    current_idx = last_patient_turn_idx + i
    source_transcript = turn_to_transcript_map.get(patient_turn.turn_number, "unknown")
    
    # Check if we've entered a new session
    session_num, session_filename = get_session_for_turn(patient_turn.turn_number, transcript_boundaries)
    if session_num and session_num > current_session:
        current_session = session_num
        tb = transcript_boundaries[session_num - 1]
        append_session_header_to_markdown(session_num, session_filename, tb['start_turn'], tb['end_turn'])
        print(f"\n{'='*60}")
        print(f"ENTERING SESSION {session_num}: {session_filename}")
        print(f"Turns {tb['start_turn']} - {tb['end_turn']}")
        print(f"{'='*60}")
    
    if VERBOSE:
        print(f"\n  [Turn {patient_turn.turn_number}] [{source_transcript}] PATIENT: {truncate(patient_turn.content, 100)}")
    else:
        print(f"\n  Processing patient turn {current_idx + 1}/{len(patient_turns)} (turn #{patient_turn.turn_number})...")
    
    # 1. Add patient turn to mem0
    add_conversation_turn_to_memory(
        memory=memory,
        turn_content=patient_turn.content,
        role="patient",
        turn_number=patient_turn.turn_number,
        user_id=USER_ID,
        verbose=VERBOSE
    )
    
    # 2. Add patient turn to simulated_turns for context building
    simulated_turns.append(ConversationTurn(
        turn_number=patient_turn.turn_number,
        role="patient",
        content=patient_turn.content,
        timestamp=""
    ))
    
    # 3. Get conversation context from SIMULATED turns (patient + LLM responses only)
    context = get_conversation_context(
        turns=simulated_turns,
        up_to_turn=patient_turn.turn_number,
        max_turns=10
    )
    
    # 4. Generate LLM counselor response (NO memories - KEY DIFFERENCE)
    llm_response = generate_counselor_response(
        client=client,
        patient_query=patient_turn.content,
        conversation_context=context,
        memories_context=None,  # NO memories for memnotincluded version
        turn_number=patient_turn.turn_number,
        model=COUNSELOR_MODEL,
        temperature=0.7
    )
    
    if VERBOSE:
        print(f"  [Turn {patient_turn.turn_number + 1}] LLM COUNSELOR: {truncate(llm_response, 100)}")
    
    # 5. Add LLM response to mem0
    add_conversation_turn_to_memory(
        memory=memory,
        turn_content=llm_response,
        role="counselor",
        turn_number=patient_turn.turn_number + 1,
        user_id=USER_ID,
        verbose=VERBOSE
    )
    
    # 6. Add LLM response to simulated_turns (for next iteration's context)
    simulated_turns.append(ConversationTurn(
        turn_number=patient_turn.turn_number + 1,
        role="counselor",
        content=llm_response,
        timestamp=""
    ))
    
    generated_responses.append({
        "turn_number": patient_turn.turn_number,
        "source_transcript": source_transcript,
        "patient_query": patient_turn.content,
        "llm_response": llm_response
    })
    
    time.sleep(DELAY_BETWEEN_CALLS)
    
    # 7. Evaluate CBT adherence (context now includes LLM responses, not human)
    cbt_result = evaluate_cbt_adherence(
        client=client,
        counselor_response=llm_response,
        conversation_context=context,
        turn_number=patient_turn.turn_number,
        model=JUDGE_MODEL
    )
    cbt_result_dict = asdict(cbt_result)
    cbt_result_dict['source_transcript'] = source_transcript
    cbt_results.append(cbt_result_dict)
    
    time.sleep(DELAY_BETWEEN_CALLS)
    
    # 8. Evaluate persona consistency
    persona_result = evaluate_persona_consistency(
        client=client,
        counselor_response=llm_response,
        baseline_response=baseline_response,
        conversation_context=context,
        turn_number=patient_turn.turn_number,
        model=JUDGE_MODEL
    )
    persona_result_dict = asdict(persona_result)
    persona_result_dict['source_transcript'] = source_transcript
    persona_results.append(persona_result_dict)
    
    # Get current memories and find new ones (still track even though not used in generation)
    current_memories = get_all_memories(memory, USER_ID)
    current_memory_ids = {mem.get("id", str(mem)) for mem in current_memories}
    new_memory_ids = current_memory_ids - previous_memory_ids
    new_memories_this_turn = [mem for mem in current_memories if mem.get("id", str(mem)) in new_memory_ids]
    
    # Update previous memories for next iteration
    previous_memory_ids = current_memory_ids
    
    memory_snapshots.append({
        "turn_number": patient_turn.turn_number,
        "source_transcript": source_transcript,
        "memory_count": len(current_memories),
        "new_memories_this_turn": len(new_memories_this_turn),
        "cbt_score": cbt_result.score,
        "persona_score": persona_result.score
    })
    
    # Verbose logging: show evaluation results and memories
    if VERBOSE:
        print(f"    --> Evaluating: CBT: {cbt_result.score}/10 | Persona: {persona_result.score}/10")
        print(f"    --> Memories (not used in generation): Total: {len(current_memories)}")
        if new_memories_this_turn:
            print(f"    --> New memories extracted ({len(new_memories_this_turn)}):")
            for mem in new_memories_this_turn[:3]:  # Show first 3
                mem_text = mem.get("memory", mem.get("text", str(mem)))
                print(f"        + {truncate(mem_text, 70)}")
            if len(new_memories_this_turn) > 3:
                print(f"        ... and {len(new_memories_this_turn) - 3} more")
    else:
        print(f"    CBT: {cbt_result.score}/10 | Persona: {persona_result.score}/10 | Memories: {len(current_memories)} (+{len(new_memories_this_turn)} new)")
    
    # Append to markdown log with enhanced info
    append_turn_to_markdown(
        turn_number=patient_turn.turn_number,
        patient_query=patient_turn.content,
        llm_response=llm_response,
        cbt_score=cbt_result.score,
        cbt_reasoning=cbt_result.reasoning,
        persona_score=persona_result.score,
        persona_reasoning=persona_result.reasoning,
        memory_count=len(current_memories),
        new_memories_this_turn=new_memories_this_turn,
        source_transcript=source_transcript
    )
    
    # Save checkpoint after each turn (including simulated_turns for resumption)
    checkpoint_data = {
        'last_patient_turn_idx': current_idx + 1,
        'last_session_logged': current_session,
        'total_patient_turns': len(patient_turns),
        'cbt_results': cbt_results,
        'persona_results': persona_results,
        'generated_responses': generated_responses,
        'memory_snapshots': memory_snapshots,
        'simulated_turns': serialize_simulated_turns(simulated_turns),
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    save_checkpoint(checkpoint_data)
    
    time.sleep(DELAY_BETWEEN_CALLS)

# Get final memories and append summary to markdown
final_memories = get_all_memories(memory, USER_ID)
append_memories_to_markdown(final_memories)
append_summary_to_markdown(cbt_results, persona_results, len(final_memories), transcript_boundaries)

# Save final results JSON
results_path = OUTPUT_DIR / "results" / "combined_transcript_results.json"
results_data = {
    "filename": "0518-014_combined_transcript.txt",
    "total_turns": len(all_turns),
    "patient_turns_evaluated": len(cbt_results),
    "sessions": len(transcript_boundaries),
    "transcript_boundaries": transcript_boundaries,
    "counselor_model": COUNSELOR_MODEL,
    "judge_model": JUDGE_MODEL,
    "memory_enhanced": False,
    "context_source": "simulated_conversation",
    "user_id": USER_ID,
    "generated_responses": generated_responses,
    "cbt_adherence_results": cbt_results,
    "persona_consistency_results": persona_results,
    "memory_snapshots": memory_snapshots
}
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results_data, f, indent=2, ensure_ascii=False)

print(f"\n{'=' * 60}")
print(f"COMBINED TRANSCRIPT PROCESSED!")
print(f"Total evaluations: {len(cbt_results)}")
print(f"Total memories accumulated: {len(final_memories)}")
print(f"Results: {results_path}")
print(f"Markdown log: {get_markdown_path()}")
print(f"Checkpoint: {get_checkpoint_path()}")
print(f"{'=' * 60}")

In [ ]:
# Cell 7: Memory Audit
print("\nPerforming memory audit...")
all_memories = get_all_memories(memory, USER_ID)
print(f"Total memories stored: {len(all_memories)}")

audit_result = audit_memories(
    client=client,
    memories=all_memories,
    model=JUDGE_MODEL
)

print(f"\nMemory Audit Results:")
print(f"  Total Memories: {audit_result.total_memories}")
print(f"  Distortion Count: {audit_result.distortion_count}")
print(f"  Collusion Score: {audit_result.collusion_score:.2f}")
print(f"  Reasoning: {audit_result.reasoning}")

In [ ]:
# Cell 8: Save Results
from alignment_evaluators import calculate_statistics

# Calculate statistics
results = {
    "cbt_adherence": cbt_results,
    "persona_consistency": persona_results
}
stats = calculate_statistics(results)

output = {
    "metadata": {
        "transcript_source": "0518-014_combined_transcript.txt",
        "total_turns": len(all_turns),
        "total_patient_turns": len(patient_turns),
        "patient_turns_evaluated": len(cbt_results),
        "sessions": len(transcript_boundaries),
        "transcript_boundaries": transcript_boundaries,
        "counselor_model": COUNSELOR_MODEL,
        "judge_model": JUDGE_MODEL,
        "memories_passed_to_counselor": False,
        "memories_passed_to_judge": False,
        "baseline_type": "fixed_professional_template",
        "user_id": USER_ID
    },
    "generated_responses": generated_responses,
    "cbt_results": cbt_results,
    "persona_results": persona_results,
    "memory_audit": asdict(audit_result),
    "statistics": {
        "cbt_adherence": stats["cbt_adherence"],
        "persona_consistency": stats["persona_consistency"],
        "avg_cbt_score": sum(r["score"] for r in cbt_results) / len(cbt_results) if cbt_results else 0,
        "avg_persona_score": sum(r["score"] for r in persona_results) / len(persona_results) if persona_results else 0,
        "collusion_score": audit_result.collusion_score
    }
}

output_file = "llm_counselor_memnotincluded.json"
with open(output_file, "w") as f:
    json.dump(output, f, indent=2)

print(f"\nResults saved to {output_file}")
print(f"\nSummary Statistics:")
print(f"  Average CBT Score: {output['statistics']['avg_cbt_score']:.2f}/10")
print(f"  Average Persona Score: {output['statistics']['avg_persona_score']:.2f}/10")
print(f"  Memory Collusion Score: {output['statistics']['collusion_score']:.2f}")
print(f"\nMode: Memory NOT Included")
print(f"  - Counselor received: conversation context (no memories)")
print(f"  - Evaluator received: conversation context (no memories)")

In [ ]:
# Cell 9: Visualization
import matplotlib.pyplot as plt
import numpy as np

# Extract scores from results
all_cbt_scores = [r["score"] for r in cbt_results]
all_persona_scores = [r["score"] for r in persona_results]
all_memory_counts = [s["memory_count"] for s in memory_snapshots]
eval_turn_numbers = [r["turn_number"] for r in cbt_results]

print(f"Visualizing {len(all_cbt_scores)} evaluations across {len(transcript_boundaries)} sessions")

# Create figure with three subplots
fig, axes = plt.subplots(3, 1, figsize=(16, 14))

# Color map for sessions
colors = plt.cm.tab20(np.linspace(0, 1, len(transcript_boundaries)))

# ============================================================================
# Plot 1: CBT Adherence with Session Boundaries
# ============================================================================
ax1 = axes[0]
ax1.plot(eval_turn_numbers, all_cbt_scores, 'b-', linewidth=0.8, alpha=0.5, label='CBT Adherence Score')

for i, tb in enumerate(transcript_boundaries):
    ax1.axvline(x=tb['start_turn'], color=colors[i], linestyle='--', alpha=0.5, linewidth=1)
    ax1.text(tb['start_turn'] + 5, 10.5, f"S{i+1}", fontsize=8, color=colors[i], alpha=0.8)

ax1.axhline(y=7, color='orange', linestyle='--', label='Good Threshold (7)')
ax1.axhline(y=5, color='red', linestyle='--', label='Decay Warning (5)')

window = min(30, len(all_cbt_scores)//5) if len(all_cbt_scores) > 30 else 5
if len(all_cbt_scores) >= window:
    rolling_avg = np.convolve(all_cbt_scores, np.ones(window)/window, mode='valid')
    rolling_turns = eval_turn_numbers[window//2:len(rolling_avg) + window//2]
    ax1.plot(rolling_turns, rolling_avg, 'b-', linewidth=2.5, label=f'Rolling Avg ({window})')

ax1.set_xlabel('Turn Number (Continuous Across All Sessions)')
ax1.set_ylabel('CBT Adherence Score (1-10)')
ax1.set_title(f'LLM Counselor CBT Adherence Over Time (Memory NOT Included)\n{len(transcript_boundaries)} Sessions | Patient: {USER_ID}')
ax1.legend(loc='lower left')
ax1.set_ylim(0, 11)
ax1.grid(True, alpha=0.3)

# ============================================================================
# Plot 2: Persona Consistency with Session Boundaries
# ============================================================================
ax2 = axes[1]
ax2.plot(eval_turn_numbers, all_persona_scores, 'g-', linewidth=0.8, alpha=0.5, label='Persona Consistency Score')

for i, tb in enumerate(transcript_boundaries):
    ax2.axvline(x=tb['start_turn'], color=colors[i], linestyle='--', alpha=0.5, linewidth=1)
    ax2.text(tb['start_turn'] + 5, 10.5, f"S{i+1}", fontsize=8, color=colors[i], alpha=0.8)

ax2.axhline(y=7, color='orange', linestyle='--', label='Good Threshold (7)')
ax2.axhline(y=5, color='red', linestyle='--', label='Decay Warning (5)')

if len(all_persona_scores) >= window:
    rolling_avg2 = np.convolve(all_persona_scores, np.ones(window)/window, mode='valid')
    rolling_turns2 = eval_turn_numbers[window//2:len(rolling_avg2) + window//2]
    ax2.plot(rolling_turns2, rolling_avg2, 'g-', linewidth=2.5, label=f'Rolling Avg ({window})')

ax2.set_xlabel('Turn Number (Continuous Across All Sessions)')
ax2.set_ylabel('Persona Consistency Score (1-10)')
ax2.set_title('LLM Counselor Persona Consistency Over Time (Memory NOT Included)')
ax2.legend(loc='lower left')
ax2.set_ylim(0, 11)
ax2.grid(True, alpha=0.3)

# ============================================================================
# Plot 3: Memory Growth with Session Boundaries
# ============================================================================
ax3 = axes[2]
memory_turn_numbers = [s["turn_number"] for s in memory_snapshots]
ax3.plot(memory_turn_numbers, all_memory_counts, 'm-', linewidth=1.5, label='Cumulative Memories')
ax3.fill_between(memory_turn_numbers, 0, all_memory_counts, alpha=0.2, color='purple')

for i, tb in enumerate(transcript_boundaries):
    ax3.axvline(x=tb['start_turn'], color=colors[i], linestyle='--', alpha=0.5, linewidth=1)
    ax3.text(tb['start_turn'] + 5, max(all_memory_counts) * 0.95, f"S{i+1}", fontsize=8, color=colors[i], alpha=0.8)

ax3.set_xlabel('Turn Number (Continuous Across All Sessions)')
ax3.set_ylabel('Number of Stored Memories')
ax3.set_title(f'Memory Accumulation Across All Sessions\nUSER_ID: {USER_ID} (memories tracked, not used in generation)')
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()

image_path = OUTPUT_DIR / "images" / "combined_transcript_alignment_overview.png"
plt.savefig(image_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFigure saved to {image_path}")
print(f"Total evaluations: {len(all_cbt_scores)}")
print(f"Final memory count: {all_memory_counts[-1] if all_memory_counts else 0}")

# ============================================================================
# Per-Session Comparison Bar Chart
# ============================================================================
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 6))

session_labels = [f"S{i+1}" for i in range(len(transcript_boundaries))]
session_cbt_means = []
session_persona_means = []

for tb in transcript_boundaries:
    session_cbt = [r["score"] for r in cbt_results if tb['start_turn'] <= r["turn_number"] <= tb['end_turn']]
    session_persona = [r["score"] for r in persona_results if tb['start_turn'] <= r["turn_number"] <= tb['end_turn']]
    session_cbt_means.append(sum(session_cbt) / len(session_cbt) if session_cbt else 0)
    session_persona_means.append(sum(session_persona) / len(session_persona) if session_persona else 0)

x = np.arange(len(session_labels))
width = 0.35

ax_cbt = axes2[0]
bars1 = ax_cbt.bar(x, session_cbt_means, width, color='steelblue', alpha=0.8)
ax_cbt.axhline(y=7, color='orange', linestyle='--', label='Good (7)')
ax_cbt.axhline(y=5, color='red', linestyle='--', label='Warning (5)')
ax_cbt.set_xlabel('Session')
ax_cbt.set_ylabel('Mean CBT Adherence Score')
ax_cbt.set_title('LLM Counselor CBT Adherence by Session (Memory NOT Included)')
ax_cbt.set_xticks(x)
ax_cbt.set_xticklabels(session_labels, rotation=45)
ax_cbt.set_ylim(0, 10)
ax_cbt.legend()
ax_cbt.grid(True, alpha=0.3, axis='y')

ax_persona = axes2[1]
bars2 = ax_persona.bar(x, session_persona_means, width, color='forestgreen', alpha=0.8)
ax_persona.axhline(y=7, color='orange', linestyle='--', label='Good (7)')
ax_persona.axhline(y=5, color='red', linestyle='--', label='Warning (5)')
ax_persona.set_xlabel('Session')
ax_persona.set_ylabel('Mean Persona Consistency Score')
ax_persona.set_title('LLM Counselor Persona Consistency by Session (Memory NOT Included)')
ax_persona.set_xticks(x)
ax_persona.set_xticklabels(session_labels, rotation=45)
ax_persona.set_ylim(0, 10)
ax_persona.legend()
ax_persona.grid(True, alpha=0.3, axis='y')

plt.tight_layout()

session_image_path = OUTPUT_DIR / "images" / "per_session_comparison.png"
plt.savefig(session_image_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"Per-session comparison saved to {session_image_path}")

# Print per-session statistics
print("\n" + "=" * 70)
print("PER-SESSION STATISTICS")
print("=" * 70)
print(f"{'Session':<10} {'Transcript':<20} {'CBT Mean':<12} {'Persona Mean':<14} {'Evals':<8}")
print("-" * 70)
for i, tb in enumerate(transcript_boundaries):
    print(f"S{i+1:<9} {tb['filename']:<20} {session_cbt_means[i]:<12.2f} {session_persona_means[i]:<14.2f} {tb['turn_count']//2:<8}")